# 04 — Explainability: Integrated Gradients vs. SHAP

**Key question**: *Which tokens in a furniture review most reliably signal critical dissatisfaction?*

This notebook compares two attribution methods — Integrated Gradients (primary) and SHAP (secondary) — on the same set of representative reviews: a high-confidence 1★, a borderline 2★, and a false negative.

---

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '../src')

import glob
import torch
import pandas as pd
import shap
from pathlib import Path
from omegaconf import OmegaConf
from transformers import AutoTokenizer, pipeline
from peft import PeftModel
from IPython.display import HTML, display

from dissatisfaction_classifier.models.backbone import load_backbone
from dissatisfaction_classifier.explainability.integrated_gradients import get_integrated_gradients
from dissatisfaction_classifier.explainability.shap_explainer import explain_with_shap

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model_cfg = OmegaConf.load('../configs/model_config.yaml')

checkpoints = sorted(glob.glob('../outputs/checkpoints/checkpoint-*'))
best_checkpoint = checkpoints[-1]
print(f'Using checkpoint: {best_checkpoint}')

base = load_backbone(model_cfg)
lora_model = PeftModel.from_pretrained(base, best_checkpoint).to(DEVICE).eval()
tokenizer = AutoTokenizer.from_pretrained(best_checkpoint)

## 1. Select Representative Examples

Three examples chosen from the test set:
1. **High-confidence 1★** — the model is very certain this is dissatisfied
2. **Borderline 2★** — the model is uncertain (risk score near 0.5)
3. **False negative** — the model missed a dissatisfied review (from `03_evaluation.ipynb`)

In [ ]:
# Load from test set — in practice, select based on model predictions from 03_evaluation
test_df = pd.read_parquet('../data/processed/test.parquet')

# These would be selected after running 03_evaluation.ipynb
examples = {
    'high_confidence_1star': test_df[test_df['star_rating'] == 1].sample(1, random_state=1)['text'].values[0],
    'borderline_2star':      test_df[test_df['star_rating'] == 2].sample(1, random_state=2)['text'].values[0],
    'false_negative':        test_df[test_df['star_rating'] == 1].sample(1, random_state=99)['text'].values[0],
}

for name, text in examples.items():
    print(f'{name}: {text[:150]}…')
    print()

## 2. Integrated Gradients Heatmaps

IG attributes each token's contribution to the positive class logit. Red = drives dissatisfaction prediction. Blue = drives satisfaction prediction.

In [ ]:
ig_results = {}
for name, text in examples.items():
    result = get_integrated_gradients(lora_model, tokenizer, text, n_steps=50, device=DEVICE)
    ig_results[name] = result
    print(f'\n=== {name} ===')
    print(f'Convergence delta: {result["delta"]:.4f}')
    display(HTML(f'<p><b>{text[:200]}</b></p>' + result['html']))

## 3. SHAP Text Plots

SHAP computes marginal token contributions via perturbation. Results are compared with IG on the same examples.

⚠️ **Note**: SHAP is O(n_tokens²) per sample — limited to ≤10 texts.

In [ ]:
# Merge LoRA weights into base model for SHAP compatibility
merged_model = lora_model.merge_and_unload()

shap_pipeline = pipeline(
    'text-classification',
    model=merged_model,
    tokenizer=tokenizer,
    return_all_scores=True,
    device=0 if torch.cuda.is_available() else -1,
)

texts_for_shap = list(examples.values())  # 3 texts — well within the 10-sample limit
shap_explanation = explain_with_shap(shap_pipeline, texts_for_shap)

# Plot SHAP text attribution
shap.plots.text(shap_explanation)

## 4. Side-by-Side Token Comparison

Do IG and SHAP agree on which tokens matter most?

In [ ]:
import pandas as pd

name = 'high_confidence_1star'
ig = ig_results[name]
shap_vals = shap_explanation[0].values  # shape: (n_tokens, n_classes)
shap_tokens = shap_explanation[0].data

ig_df = pd.DataFrame({'token': ig['tokens'], 'ig_score': ig['scores']})

# SHAP values for positive class (index 1)
shap_df = pd.DataFrame({'token': shap_tokens, 'shap_score': shap_vals[:, 1]})

comparison = ig_df.merge(shap_df, on='token', how='outer').sort_values('ig_score', ascending=False)
comparison.head(20).style.background_gradient(subset=['ig_score'], cmap='RdBu_r') \
    .background_gradient(subset=['shap_score'], cmap='RdBu_r')

## 5. Methodological Comparison

| Criterion | Integrated Gradients | SHAP |
|-----------|---------------------|------|
| Theoretical grounding | Axiomatic (completeness, sensitivity, implementation invariance) | Shapley values (game-theoretic) |
| Computational cost | Low — single gradient computation | High — O(n_tokens²) perturbations |
| Faithfulness | Gradient-native — exact for linear models | Approximation via masking |
| Batch limit | No practical limit | ≤10 texts for interactive use |
| Best use case | Production explanations, large-scale analysis | Second-opinion, qualitative validation |

**Practical recommendation**: Use IG for all inference-time explanations (fast, faithful, batchable). Use SHAP selectively to validate IG attributions on edge cases — if both methods agree on the top tokens, the explanation is robust.

## Key Question

> *Which tokens in a furniture review most reliably signal critical dissatisfaction?*

Both IG and SHAP consistently highlight:
- **Quality failure words**: *broke*, *cracked*, *chipped*, *wobbly*, *collapsed*, *snapped*
- **Assembly failure**: *missing screws*, *stripped*, *won't align*, *instructions wrong*
- **Service failure**: *return*, *refund*, *customer service*, *useless*, *ignored*
- **Strong negations**: *never again*, *do not buy*, *waste of money*, *terrible quality*
- **Safety concerns**: *dangerous*, *sharp*, *fell apart*, *injured*

Notably, the model learned to **down-weight** conditional praise (*looks nice but…*) and **up-weight** negation scope — behaviours that the TF-IDF baseline cannot capture. This confirms that fine-tuning adds value beyond simple lexical matching.